# Silver layer

## Customer additional information (birthdate and gender)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, trim, upper, now, isnull, current_date, max, min, isnotnull, length, lag, date_add, lead, isnull, ifnull
from pyspark.sql.types import *

In [0]:
df = spark.read.table("db_project.bronze.erp_cust_az12")
df.display()

In [0]:
test_id = df.select(
    min(df.CID).alias("min CID length"),
    max(df.CID).alias("max CID length"),
    F.avg(length(df.CID)).alias("avg CID length")
)
test_id.display()

In [0]:
df.select(length(df.CID)).distinct().display()

In [0]:
df = df.withColumn("CID", 
              F.when(length(df.CID) == 13, F.substring(df.CID, 4, length(df.CID)))
              .when(length(df.CID) == 10, df.CID)
              .otherwise("n/a")
)
df.display()

In [0]:
df.select(df.GEN).distinct().display()

In [0]:
df = df.withColumn("gen", 
              F.when(F.trim(df.gen).isin("Female", "F", "f"), "Female")
              .when(F.trim(df.gen).isin("Male", "M", "m"), "Male")
              .otherwise("n/a")
)
df.display()

In [0]:
df.select(
    min(df.BDATE).alias("min BDATE"),
    max(df.BDATE).alias("max BDATE")
).display()

In [0]:
df = df.withColumn("bdate", 
              F.when(df.bdate > current_date(), None)
              .otherwise(df.BDATE)
)
df.display()

In [0]:
df.write.mode("overwrite").option("overwriteSchema", True).format("delta").saveAsTable("db_project.silver.erp_cust_az12")